# Audio Feature Extraction
## Notebook Summary

This notebook downloads Apple Music preview clips, extracts computational audio features using Essentia machine-learning models, and merges the resulting features with Billboard lifecycle metrics and Apple metadata. The final output is the analysis-ready dataset used throughout the remainder of the project.

### Inputs

* `usable_apple_matches_v2.csv`
* `billboard_song_lifecycle.csv`
* Apple preview clips
* Essentia model files

### Outputs

* `audio_features_full.csv`
* `audio_features_full_with_tempo_adjusted.csv`
* `final_df.csv`
* `final_df_FULL_BACKUP.csv`

### Key Contributions

* Extract tempo, loudness, spectral, valence, arousal, and genre features.
* Validate feature quality and apply tempo corrections.
* Construct the final integrated dataset for clustering and analysis.


## 0) Setup + Load

- Load Apple metadata and preview download results.
- Load genre metadata and TensorFlow / Essentia models.
- Prepare resources required for audio feature extraction.

In [1]:
import os
from pathlib import Path
import json
import time
import requests
import re

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from tqdm.notebook import tqdm

import essentia
import essentia.standard as es

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

In [11]:
top40 = pd.read_csv("../data/intermediate/top40_billboard_dataset.csv") # 
apple_df = pd.read_csv("../data/intermediate/usable_apple_matches_v2.csv")  # only after current job finishes

# Folder for downloaded Apple preview clips
PREVIEW_DIR = Path("../preview_clips")
PREVIEW_DIR.mkdir(exist_ok=True)

# Folder where Essentia model files are stored
MODEL_DIR = Path("../essentia_models")

usable_apple_df = pd.read_csv("../data/intermediate/usable_apple_matches_v2.csv")

In [9]:
with open(
    "../essentia_models/genre_discogs400-discogs-effnet-1.json"
) as f:
    metadata = json.load(f)

labels = metadata["classes"]

print(len(labels))

400


### Load Essentia Models

In [12]:
# MusiCNN embedder used for valence/arousal model
musicnn_embedder = es.TensorflowPredictMusiCNN(
    graphFilename=str(MODEL_DIR / "msd-musicnn-1.pb"),
    output="model/dense/BiasAdd"
)

# Valence/arousal model
deam_model = es.TensorflowPredict2D(
    graphFilename=str(MODEL_DIR / "deam-msd-musicnn-2.pb"),
    input="model/Placeholder",
    output="model/Identity"
)

# EffNet embedder used for Discogs genre model
effnet_embedder = es.TensorflowPredictEffnetDiscogs(
    graphFilename=str(MODEL_DIR / "discogs-effnet-bs64-1.pb"),
    output="PartitionedCall:1"
)

# Discogs400 genre classifier
genre_model = es.TensorflowPredict2D(
    graphFilename=str(MODEL_DIR / "genre_discogs400-discogs-effnet-1.pb"),
    input="serving_default_model_Placeholder",
    output="PartitionedCall:0"
)

print("Models loaded.")

Models loaded.


[   INFO   ] TensorflowPredict: Successfully loaded graph file: `../essentia_models/msd-musicnn-1.pb`
[   INFO   ] TensorflowPredict: Successfully loaded graph file: `../essentia_models/deam-msd-musicnn-2.pb`
[   INFO   ] TensorflowPredict: Successfully loaded graph file: `../essentia_models/discogs-effnet-bs64-1.pb`
[   INFO   ] TensorflowPredict: Successfully loaded graph file: `../essentia_models/genre_discogs400-discogs-effnet-1.pb`


Danceability was tested but excluded from the final feature set because manual inspection showed unreliable predictions on preview clips.

## 1) Helper Functions

In [ ]:
def safe_filename(text):
    """
    Convert song/artist text into a safe filename.
    """
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = text.strip("_")
    return text[:120]

In [ ]:
def download_preview(row, preview_dir=PREVIEW_DIR, timeout=15):
    """
    Download one Apple preview clip from preview_url.
    Returns:
        preview_file path
        download_status
    """
    song = row["search_song"]
    artist = row["search_artist"]
    url = row["preview_url"]

    if pd.isna(url) or url == "":
        return None, "missing_url"

    apple_id = row.get("apple_track_id", row.name)

    filename = (
        f"{safe_filename(artist)}__"
        f"{safe_filename(song)}__"
        f"{apple_id}.m4a"
    )

    filepath = preview_dir / filename

    # Skip if already downloaded
    if filepath.exists() and filepath.stat().st_size > 0:
        return str(filepath), "already_exists"

    try:
        response = requests.get(url, timeout=timeout)

        if response.status_code != 200:
            return None, f"status_{response.status_code}"

        with open(filepath, "wb") as f:
            f.write(response.content)

        if filepath.stat().st_size == 0:
            filepath.unlink(missing_ok=True)
            return None, "empty_file"

        return str(filepath), "downloaded"

    except Exception as e:
        return None, f"error_{type(e).__name__}"

In [ ]:
def get_top_genres(labels, genre_mean, n=5):
    """
    Return top n detailed Discogs genre predictions.
    """
    top_indices = np.argsort(genre_mean)[::-1][:n]

    genre_features = {}

    for rank, idx in enumerate(top_indices, start=1):
        genre_features[f"genre_{rank}"] = labels[idx]
        genre_features[f"genre_{rank}_score"] = float(genre_mean[idx])

    return genre_features


In [ ]:
def clean_genre_family_name(family):
    """
    Convert genre family text into clean column suffix.
    """
    family = family.lower()
    family = family.replace("&", "and")
    family = family.replace("/", "_")
    family = re.sub(r"[^a-z0-9]+", "_", family)
    family = family.strip("_")
    return family


def aggregate_genre_families(labels, genre_mean):
    """
    Aggregate 400 detailed Discogs genre probabilities into broad families.
    Example:
        Rock---Alternative Rock -> genre_family_rock
        Hip Hop---Trap -> genre_family_hip_hop
    """
    family_scores = {}

    for label, score in zip(labels, genre_mean):
        family = label.split("---")[0]
        family_clean = clean_genre_family_name(family)
        col = f"genre_family_{family_clean}"

        family_scores[col] = family_scores.get(col, 0) + float(score)

    return family_scores

In [ ]:
def adjust_tempo_final(row):
    """
    Create adjusted tempo while keeping raw tempo.

    Logic:
    - Raw Essentia tempo often suffers from half/double-time ambiguity.
    - For Billboard tracks, 145–180 BPM is often double-time.
    - But strong electronic/rock tracks may genuinely be fast, so we keep those.
    """
    bpm = row["tempo_bpm"]

    electronic = row.get("genre_family_electronic", 0)
    rock = row.get("genre_family_rock", 0)

    adjusted = bpm
    reason = "unchanged"

    # Likely double-time region
    if 145 <= bpm <= 180:
        if electronic >= 0.80 or rock >= 0.80:
            adjusted = bpm
            reason = "kept_high_bpm_strong_electronic_or_rock"
        else:
            adjusted = bpm / 2
            reason = "halved_145_180_likely_double_time"

    # Extremely high tempos are usually double-time in this dataset
    elif bpm > 180:
        adjusted = bpm / 2
        reason = "halved_over_180_extreme_bpm"

    return pd.Series({
        "tempo_bpm_raw": bpm,
        "tempo_bpm_adjusted": adjusted,
        "tempo_adjustment_reason": reason
    })

## 2) Audio Feature Extraction Function
- Define the primary feature extraction pipeline.
- Extract tempo, loudness, spectral features, valence, arousal, and genres.
- Return a standardized feature record for each preview clip.

In [ ]:
def extract_audio_features(filename, labels):
    """
    Extract final audio features from one Apple preview clip.

    Important:
    - Danceability is intentionally excluded because both tested models saturated.
    - Exact key is intentionally excluded because it is noisy and not analytically useful.
    - Mode and key_strength are kept as lightweight harmonic features.
    """
    features = {
        "preview_filename": filename
    }

    try:
        # =========================
        # Load audio at two sample rates
        # =========================

        # 44.1k audio for traditional audio descriptors
        audio_44100 = es.MonoLoader(
            filename=filename,
            sampleRate=44100
        )()

        # 16k audio for TensorFlow models
        audio_16000 = es.MonoLoader(
            filename=filename,
            sampleRate=16000,
            resampleQuality=4
        )()

        features["preview_duration_sec"] = float(len(audio_44100) / 44100)

        # =========================
        # Rhythm / tempo
        # =========================
        rhythm = es.RhythmExtractor2013(method="multifeature")
        bpm, beats, beat_confidence, _, _ = rhythm(audio_44100)

        # Keep raw tempo. Adjustment will happen later during cleaning.
        features["tempo_bpm"] = float(bpm)
        features["beat_confidence"] = float(beat_confidence)

        # =========================
        # Loudness
        # =========================
        features["rms_loudness"] = float(es.RMS()(audio_44100))

        # =========================
        # Mode / key strength
        # =========================
        key_extractor = es.KeyExtractor()
        key, mode, key_strength = key_extractor(audio_44100)

        # Exact key is not saved because it is noisy and hard to interpret.
        features["mode"] = mode
        features["key_strength"] = float(key_strength)

        # =========================
        # Spectral / timbre features
        # =========================
        frame_size = 2048
        hop_size = 1024

        window = es.Windowing(type="hann")
        spectrum = es.Spectrum()
        centroid = es.Centroid(range=22050)
        rolloff = es.RollOff(sampleRate=44100)

        centroids = []
        rolloffs = []

        for frame in es.FrameGenerator(
            audio_44100,
            frameSize=frame_size,
            hopSize=hop_size
        ):
            spec = spectrum(window(frame))
            centroids.append(centroid(spec))
            rolloffs.append(rolloff(spec))

        features["spectral_centroid_mean"] = float(np.mean(centroids))
        features["spectral_rolloff_mean"] = float(np.mean(rolloffs))

        # =========================
        # Valence / arousal
        # =========================
        musicnn_embeddings = musicnn_embedder(audio_16000)

        deam_preds = deam_model(musicnn_embeddings)
        deam_mean = deam_preds.mean(axis=0)

        features["valence"] = float(deam_mean[0])
        features["arousal"] = float(deam_mean[1])

        # =========================
        # Genre features
        # =========================
        effnet_embeddings = effnet_embedder(audio_16000)

        genre_preds = genre_model(effnet_embeddings)
        genre_mean = genre_preds.mean(axis=0)

        # Top 5 detailed genres
        top_genre_features = get_top_genres(
            labels=labels,
            genre_mean=genre_mean,
            n=5
        )
        features.update(top_genre_features)

        # Broad genre-family probabilities
        family_features = aggregate_genre_families(
            labels=labels,
            genre_mean=genre_mean
        )
        features.update(family_features)

        # Convert any leftover numpy scalar values to regular floats
        for k, v in list(features.items()):
            if isinstance(v, np.generic):
                features[k] = float(v)

        features["audio_feature_error"] = None

        return features

    except Exception as e:
        return {
            "preview_filename": filename,
            "audio_feature_error": str(e)
        }

## 3) Download Previews

- Apple preview clips were downloaded from the validated Apple metadata dataset.
- Download results were tracked to confirm which songs had usable local audio files.
- These downloaded clips became the input files for Essentia audio feature extraction.

In [ ]:
download_results = []

total = len(usable_apple_df)
start = time.time()

for i, (idx, row) in enumerate(
    tqdm(usable_apple_df.iterrows(), total=total),
    start=1
):
    preview_file, download_status = download_preview(row)

    download_results.append({
        "index": idx,
        "search_song": row.get("search_song"),
        "search_artist": row.get("search_artist"),
        "apple_track_id": row.get("apple_track_id"),
        "preview_file": preview_file,
        "download_status": download_status
    })

    # Save checkpoint every 100 songs
    if i % 100 == 0:
        checkpoint_df = pd.DataFrame(download_results)
        checkpoint_df.to_csv("../data/intermediate/preview_download_checkpoint.csv", index=False)

        elapsed = time.time() - start
        rate = i / elapsed
        remaining = total - i
        eta_min = remaining / rate / 60

        print(
            f"Downloaded/checkpointed {i}/{total} "
            f"({i / total:.1%}) | "
            f"Rate: {rate:.2f} songs/sec | "
            f"ETA: {eta_min:.1f} min"
        )

    # Gentle pause so we don't hammer Apple preview URLs
    time.sleep(0.05)

download_df = pd.DataFrame(download_results)

download_df.to_csv(
    "../data/intermediate/preview_download_results.csv",
    index=False
)

usable_apple_with_files = usable_apple_df.reset_index().merge(
    download_df,
    on="index",
    how="left"
)

usable_apple_with_files.to_csv(
    "../data/intermediate/usable_apple_matches_with_files.csv",
    index=False
)

print("Done downloading.")
print(download_df["download_status"].value_counts(dropna=False))
print("Usable with files:", usable_apple_with_files["preview_file"].notna().mean())

  0%|          | 0/3783 [00:00<?, ?it/s]

Downloaded/checkpointed 100/3783 (2.6%) | Rate: 18.01 songs/sec | ETA: 3.4 min
Downloaded/checkpointed 200/3783 (5.3%) | Rate: 6.18 songs/sec | ETA: 9.7 min
Downloaded/checkpointed 300/3783 (7.9%) | Rate: 4.99 songs/sec | ETA: 11.6 min
Downloaded/checkpointed 400/3783 (10.6%) | Rate: 4.58 songs/sec | ETA: 12.3 min
Downloaded/checkpointed 500/3783 (13.2%) | Rate: 4.29 songs/sec | ETA: 12.7 min
Downloaded/checkpointed 600/3783 (15.9%) | Rate: 4.08 songs/sec | ETA: 13.0 min
Downloaded/checkpointed 700/3783 (18.5%) | Rate: 3.95 songs/sec | ETA: 13.0 min
Downloaded/checkpointed 800/3783 (21.1%) | Rate: 3.85 songs/sec | ETA: 12.9 min
Downloaded/checkpointed 900/3783 (23.8%) | Rate: 3.71 songs/sec | ETA: 12.9 min
Downloaded/checkpointed 1000/3783 (26.4%) | Rate: 3.62 songs/sec | ETA: 12.8 min
Downloaded/checkpointed 1100/3783 (29.1%) | Rate: 3.56 songs/sec | ETA: 12.6 min
Downloaded/checkpointed 1200/3783 (31.7%) | Rate: 3.46 songs/sec | ETA: 12.4 min
Downloaded/checkpointed 1300/3783 (34.4%)

In [ ]:
download_df["download_status"].value_counts(dropna=False)

download_status
downloaded           3682
already_exists        100
error_ReadTimeout       1
Name: count, dtype: int64

One timed-out preview was later manually downloaded, resulting in 3,783 songs available for feature extraction.

## 4a) Full Audio Feature Extraction

- The finalized feature extraction function was applied to all successfully downloaded preview clips.
- Extracted features included tempo, loudness, spectral brightness, valence, arousal, mode, and genre predictions.
- The full feature dataset was saved for validation and downstream analysis.

In [ ]:
## Extract Audio Features w/ checkpoints and status

valid_files_df = usable_apple_with_files[
    usable_apple_with_files["preview_file"].notna()
].copy()

valid_files_df = valid_files_df.reset_index(drop=True)

print("Files to extract:", len(valid_files_df))

Files to extract: 3783


In [ ]:
audio_results = []

total = len(valid_files_df)
start = time.time()

for i, row in enumerate(
    tqdm(valid_files_df.itertuples(index=False), total=total),
    start=1
):
    features = extract_audio_features(
        filename=row.preview_file,
        labels=labels
    )

    # Add identifiers so we can merge later
    features["apple_track_id"] = getattr(row, "apple_track_id_x", None)
    features["search_song"] = getattr(row, "search_song_x", None)
    features["search_artist"] = getattr(row, "search_artist_x", None)
    features["preview_file"] = row.preview_file

    audio_results.append(features)

    # Save checkpoint every 100 songs
    if i % 100 == 0:
        checkpoint_df = pd.DataFrame(audio_results)

        checkpoint_df.to_csv(
            "../data/intermediate/audio_features_checkpoint.csv",
            index=False
        )

        elapsed = time.time() - start
        rate = i / elapsed
        remaining = total - i
        eta_min = remaining / rate / 60

        errors = checkpoint_df["audio_feature_error"].notna().sum()

        print(
            f"Extracted/checkpointed {i}/{total} "
            f"({i / total:.1%}) | "
            f"Rate: {rate:.2f} songs/sec | "
            f"ETA: {eta_min:.1f} min | "
            f"Errors: {errors}"
        )

audio_features_full = pd.DataFrame(audio_results)

audio_features_full.to_csv(
    "../data/intermediate/audio_features_full.csv",
    index=False
)

print("Done extracting.")
print("Shape:", audio_features_full.shape)
print(audio_features_full["audio_feature_error"].value_counts(dropna=False))

  0%|          | 0/3783 [00:00<?, ?it/s]

Extracted/checkpointed 100/3783 (2.6%) | Rate: 0.71 songs/sec | ETA: 86.9 min | Errors: 0
Extracted/checkpointed 200/3783 (5.3%) | Rate: 0.70 songs/sec | ETA: 85.0 min | Errors: 0
Extracted/checkpointed 300/3783 (7.9%) | Rate: 0.69 songs/sec | ETA: 83.8 min | Errors: 0
Extracted/checkpointed 400/3783 (10.6%) | Rate: 0.68 songs/sec | ETA: 82.4 min | Errors: 0
Extracted/checkpointed 500/3783 (13.2%) | Rate: 0.68 songs/sec | ETA: 80.5 min | Errors: 0
Extracted/checkpointed 600/3783 (15.9%) | Rate: 0.68 songs/sec | ETA: 78.3 min | Errors: 0
Extracted/checkpointed 700/3783 (18.5%) | Rate: 0.68 songs/sec | ETA: 75.9 min | Errors: 0
Extracted/checkpointed 800/3783 (21.1%) | Rate: 0.68 songs/sec | ETA: 73.3 min | Errors: 0
Extracted/checkpointed 900/3783 (23.8%) | Rate: 0.68 songs/sec | ETA: 70.7 min | Errors: 0
Extracted/checkpointed 1000/3783 (26.4%) | Rate: 0.68 songs/sec | ETA: 68.2 min | Errors: 0
Extracted/checkpointed 1100/3783 (29.1%) | Rate: 0.68 songs/sec | ETA: 65.5 min | Errors: 0


## 4b) Tempo adjustment

- Review extracted tempo values for common double-time or half-time errors.
- Apply a rule-based correction using genre context.
- Save adjusted tempo values for downstream PCA and clustering.

In [ ]:
tempo_adjustments = audio_features_full.apply(
    adjust_tempo_final,
    axis=1
)

audio_features_full = pd.concat(
    [audio_features_full, tempo_adjustments],
    axis=1
)

audio_features_full.to_csv(
    "../data/intermediate/audio_features_full_with_tempo_adjusted.csv",
    index=False
)

audio_features_full[
    ["tempo_bpm_raw", "tempo_bpm_adjusted"]
].describe()

,tempo_bpm_raw,tempo_bpm_adjusted
count,3783.000000,3783.000000
mean,117.251375,108.876222
std,25.369502,23.781746
min,59.921703,59.921703
25%,95.329132,88.773232
50%,117.921303,106.933563
75%,136.425705,128.058739
max,184.570312,178.205811


In [ ]:
audio_features_full["tempo_adjustment_reason"].value_counts()

tempo_adjustment_reason
unchanged                                  3191
halved_145_180_likely_double_time           398
kept_high_bpm_strong_electronic_or_rock     187
halved_over_180_extreme_bpm                   7
Name: count, dtype: int64

## 5) Final Dataset Merge

- Billboard lifecycle metrics, Apple metadata, and extracted audio features were merged into one analysis-ready dataset `final_df.csv`.
- The merge preserved chart performance, music metadata, and computational audio features for each matched song.
- The resulting dataset became the source for PCA, clustering, and final poster figures.

In [ ]:
top40 = pd.read_csv("../data/intermediate/top40_billboard_dataset.csv")
apple = pd.read_csv("../data/intermediate/usable_apple_matches_v2.csv")
audio = pd.read_csv("../data/intermediate/audio_features_full_with_tempo_adjusted.csv")

print("top40:", top40.shape)
print("apple:", apple.shape)
print("audio:", audio.shape)

top40: (4078, 17)
apple: (3783, 24)
audio: (3783, 44)


In [ ]:
# Merge back to Billboard dataset.
# This assumes search_song/search_artist correspond to Billboard Song/Artist.
final_analysis_df = top40.merge(
    apple_audio,
    left_on=["Song", "Artist"],
    right_on=["search_song", "search_artist"],
    how="left"
)

print("final_analysis_df:", final_analysis_df.shape)
final_analysis_df.head()

final_analysis_df: (4078, 82)


,Song,Artist,first_chart_date,last_chart_date,entry_rank,peak_rank,total_weeks,top10_weeks,top40_weeks,avg_rank,...,genre_family_non_music,genre_family_pop,genre_family_reggae,genre_family_rock,genre_family_stage_and_screen,audio_feature_error,preview_file,tempo_bpm_raw,tempo_bpm_adjusted,tempo_adjustment_reason
0,"""1 Step Forward, 3 Steps Back""",Olivia Rodrigo,2021-06-02,2021-06-23,19,19,4,0,2,54.250000,...,0.014486,0.448652,0.005238,0.152280,0.011177,NaN,preview_clips/olivia_rodrigo__1_step_forward_3...,172.265625,86.132812,halved_145_180_likely_double_time
1,"""1, 2 Step""",Ciara Featuring Missy Elliott,2005-01-05,2005-07-20,2,2,39,12,24,19.448276,...,0.000528,0.070866,0.003168,0.007899,0.000968,NaN,preview_clips/ciara_featuring_missy_elliott__1...,116.173401,116.173401,unchanged
2,"""1, 2, 3, 4""",Plain White T's,2009-02-04,2009-06-17,91,34,20,0,5,47.800000,...,0.025339,0.083516,0.023637,1.199483,0.002437,NaN,preview_clips/plain_white_t_s__1_2_3_4__160588...,89.632843,89.632843,unchanged
3,"""10,000 Hours""",Dan + Shay & Justin Bieber,2019-10-16,2020-05-06,4,4,30,14,28,15.933333,...,0.026035,0.675385,0.016751,0.140071,0.005422,NaN,preview_clips/dan_shay_justin_bieber__10_000_h...,178.205811,178.205811,kept_high_bpm_strong_electronic_or_rock
4,"""Awful, Beautiful Life""",Darryl Worley,2005-01-05,2005-03-23,49,30,20,0,2,63.583333,...,0.025739,0.228829,0.000909,0.723153,0.004464,NaN,preview_clips/darryl_worley__awful_beautiful_l...,112.451279,112.451279,unchanged


In [ ]:
# Convert dates
final_analysis_df["first_chart_date"] = pd.to_datetime(
    final_analysis_df["first_chart_date"],
    errors="coerce"
)

final_analysis_df["peak_date"] = pd.to_datetime(
    final_analysis_df["peak_date"],
    errors="coerce"
)

final_analysis_df["apple_release_date"] = pd.to_datetime(
    final_analysis_df["apple_release_date"],
    errors="coerce"
)

# Remove timezone information from all date columns

final_analysis_df["first_chart_date"] = (
    final_analysis_df["first_chart_date"]
    .dt.tz_localize(None)
)

final_analysis_df["peak_date"] = (
    final_analysis_df["peak_date"]
    .dt.tz_localize(None)
)

final_analysis_df["apple_release_date"] = (
    final_analysis_df["apple_release_date"]
    .dt.tz_localize(None)
)

# Release-to-chart lag
final_analysis_df["days_release_to_chart"] = (
    final_analysis_df["first_chart_date"] -
    final_analysis_df["apple_release_date"]
).dt.days

# Release-to-peak lag
final_analysis_df["days_release_to_peak"] = (
    final_analysis_df["peak_date"] -
    final_analysis_df["apple_release_date"]
).dt.days

## 6) Export Final Dataset

Final Dataset Summary

- Merge Billboard lifecycle metrics, Apple metadata, and extracted audio features.
- Construct the final analysis-ready dataset.
- Verify dimensions and export the completed dataset.

In [ ]:
final_analysis_df.to_csv(
    "../data/final/final_df.csv",
    index=False
)

print("Saved final_df.csv")
print(final_analysis_df.shape)

Saved final_df.csv
(4078, 84)
